# LeetCode 277 — Find the Celebrity bằng Python

Link đề: https://leetcode.com/problems/find-the-celebrity/ · Độ khó: **Medium** · Chủ đề: Graph, Two Pointers, Interactive

Skeleton LeetCode:

```python
# The knows API is already defined for you.
# return a bool, whether a knows b
# def knows(a: int, b: int) -> bool:

class Solution:
    def findCelebrity(self, n: int) -> int:
        ...
```

Ghi chú: đây là bài **Premium** của LeetCode; đề bài viết theo hiểu biết về bài (và bản mô tả đã tải), nên đối chiếu lại nếu có tài khoản Premium. Hàm `knows(a, b)` do hệ thống chấm cung cấp; trong notebook ta **tự dựng** nó từ ma trận quan hệ `graph` và **đếm số lần gọi**. Mọi đáp án được kiểm chứng bằng code và đối chiếu với một lời giải độc lập đọc thẳng ma trận.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: có `n` người (nhãn `0..n−1`); **người nổi tiếng (celebrity)** là người mà **mọi người khác đều biết** nhưng **người đó không biết ai cả**. Chỉ được hỏi dạng "A có biết B không?" qua hàm `knows(a, b)`; tìm người nổi tiếng (hoặc `-1` nếu không có) với **số câu hỏi ít nhất** về mặt tiệm cận (follow up: tối đa `3n` lần gọi).
2. Biết **tự dựng hàm `knows`** từ ma trận quan hệ và **đếm số lần gọi** để kiểm tra giới hạn gọi API.
3. Chứng minh được **tính chất then chốt**: có **nhiều nhất một** người nổi tiếng; và **mỗi câu hỏi `knows(a, b)` loại được đúng một người** khỏi danh sách ứng viên: nếu `a` biết `b` thì `a` không phải người nổi tiếng; nếu `a` không biết `b` thì `b` không phải người nổi tiếng.
4. Đi từ **kiểm tra từng ứng viên (`O(n²)` lần gọi)** → **đếm bậc vào/ra của mọi cặp (`n(n−1)` lần gọi)** → **loại dần ứng viên rồi kiểm chứng (`≤ 3n` lần gọi, cách tham khảo)** → **ngăn xếp loại ứng viên** → **hai con trỏ từ hai đầu**.
5. Tránh các lỗi kinh điển: quên bước kiểm chứng sau khi chọn ứng viên, chỉ kiểm tra một chiều (người nổi tiếng không biết ai **và** ai cũng biết người đó), cập nhật ứng viên sai hướng, và hỏi `knows(i, i)`.

## 2. Đề bài & ví dụ

Giả sử bạn ở một bữa tiệc có `n` người, đánh nhãn từ `0` đến `n − 1`, trong đó **có thể** tồn tại một người nổi tiếng. Định nghĩa: **người nổi tiếng** là người mà **tất cả `n − 1` người còn lại đều biết**, nhưng **người đó không biết ai trong số họ**.

Bạn muốn tìm ra người nổi tiếng hoặc xác nhận không có ai như vậy. Bạn chỉ được hỏi dạng: "Chào A, bạn có biết B không?" để biết A có biết B không. Hãy tìm người nổi tiếng (hoặc xác nhận không có) bằng **số câu hỏi ít nhất** (theo nghĩa tiệm cận).

Bạn được cho số nguyên `n` và hàm trợ giúp `bool knows(a, b)` cho biết `a` có biết `b` không. Cài đặt hàm `int findCelebrity(n)`. Sẽ có **đúng một** người nổi tiếng nếu họ có mặt tại bữa tiệc.

Trả về nhãn của người nổi tiếng nếu có, ngược lại trả `-1`.

Lưu ý: ma trận `graph` kích thước `n × n` ở đầu vào **không** truy cập trực tiếp được; chỉ qua `knows`. `graph[i][j] == 1` nghĩa là người `i` biết người `j`; `graph[i][j] == 0` nghĩa là `i` không biết `j`.

**Ví dụ 1**

- Input: `graph = [[1,1,0],[0,1,0],[1,1,1]]`
- Output: `1`
- Giải thích: người `1` là người nổi tiếng vì `0` và `2` đều biết `1` còn `1` không biết ai.

**Ví dụ 2**

- Input: `graph = [[1,0,1],[1,1,0],[0,1,1]]`
- Output: `-1`
- Giải thích: không có người nổi tiếng.

**Ràng buộc:** `n == graph.length == graph[i].length`; `2 <= n <= 100`; `graph[i][j]` là `0` hoặc `1`; `graph[i][i] == 1`.

**Follow up:** nếu số lần gọi `knows` tối đa cho phép là `3 * n`, bạn có tìm được lời giải không vượt quá giới hạn đó?

In [1]:
from typing import List
import random

examples = [
    ([[1, 1, 0], [0, 1, 0], [1, 1, 1]], 1),
    ([[1, 0, 1], [1, 1, 0], [0, 1, 1]], -1),
]
for graph, expected in examples:
    print(f"graph={graph}, kỳ vọng {expected}")

graph=[[1, 1, 0], [0, 1, 0], [1, 1, 1]], kỳ vọng 1
graph=[[1, 0, 1], [1, 1, 0], [0, 1, 1]], kỳ vọng -1


## 3. Kiến thức nền

### 3.1. Ma trận quan hệ và đồ thị có hướng

`graph[i][j] = 1` nghĩa là "`i` biết `j`" — một **cạnh có hướng** `i → j`. Người nổi tiếng `c` có: **mọi cạnh đi vào** `c` (`graph[i][c] = 1` với `i ≠ c`) và **không có cạnh đi ra** tới ai khác (`graph[c][j] = 0` với `j ≠ c`). Ta nói bậc vào `n − 1` và bậc ra `0` (không tính cạnh tự biết mình `graph[i][i] = 1`).

In [2]:
graph = [[1, 1, 0], [0, 1, 0], [1, 1, 1]]
n = len(graph)
for c in range(n):
    out_deg = sum(graph[c][j] for j in range(n) if j != c)
    in_deg = sum(graph[i][c] for i in range(n) if i != c)
    print(f"người {c}: biết {out_deg} người khác, được {in_deg} người khác biết", "<- người nổi tiếng" if out_deg == 0 and in_deg == n - 1 else "")

người 0: biết 1 người khác, được 1 người khác biết 
người 1: biết 0 người khác, được 2 người khác biết <- người nổi tiếng
người 2: biết 2 người khác, được 0 người khác biết 


### 3.2. Dựng hàm `knows` và đếm số lần gọi

Đề cho `knows(a, b)` có sẵn (ta không thấy ma trận). Để test, ta dựng một lớp `Party` giữ ma trận, cung cấp `knows` và **đếm số lần gọi**. Đếm giúp kiểm tra giới hạn `3n` của follow up.

In [3]:
class Party:
    """Mô phỏng bữa tiệc: chỉ cho phép hỏi qua knows(a, b) và đếm số lần hỏi."""

    def __init__(self, graph):
        self._graph = graph
        self.n = len(graph)
        self.calls = 0
        self.log = []

    def knows(self, a, b):
        assert 0 <= a < self.n and 0 <= b < self.n, (a, b)
        assert a != b, "không nên hỏi một người có biết chính mình không"
        self.calls += 1
        self.log.append((a, b))
        return bool(self._graph[a][b])


party = Party([[1, 1, 0], [0, 1, 0], [1, 1, 1]])
print(party.knows(0, 1), party.knows(1, 2), "| số lần gọi:", party.calls)

True False | số lần gọi: 2


### 3.3. Tính duy nhất của người nổi tiếng

**Có nhiều nhất một người nổi tiếng.** Giả sử `c1 ≠ c2` đều là người nổi tiếng. Vì `c1` là người nổi tiếng nên `c2` (người khác) biết `c1`; nhưng `c2` là người nổi tiếng nên `c2` không biết ai: mâu thuẫn. Vậy không thể có hai người nổi tiếng. Đề cũng nói rõ "đúng một nếu có".

In [4]:
def has_celebrity_bruteforce(graph):
    n = len(graph)
    return [c for c in range(n)
            if all(graph[c][j] == 0 for j in range(n) if j != c) and all(graph[i][c] == 1 for i in range(n) if i != c)]


random.seed(1)
for _ in range(2000):
    n = random.randint(2, 6)
    g = [[1 if i == j else random.randint(0, 1) for j in range(n)] for i in range(n)]
    assert len(has_celebrity_bruteforce(g)) <= 1
print("PASS: trong 2000 đồ thị ngẫu nhiên không có đồ thị nào có hai người nổi tiếng")

PASS: trong 2000 đồ thị ngẫu nhiên không có đồ thị nào có hai người nổi tiếng


### 3.4. Mỗi câu hỏi loại được một người

Xét một câu hỏi `knows(a, b)` với `a ≠ b`:

- nếu **có** (`a` biết `b`): `a` **không thể** là người nổi tiếng (người nổi tiếng không biết ai); `b` vẫn có thể là người nổi tiếng;
- nếu **không** (`a` không biết `b`): `b` **không thể** là người nổi tiếng (mọi người đều phải biết người nổi tiếng); `a` vẫn có thể là người nổi tiếng.

Vậy **một câu hỏi luôn loại đúng một trong hai người** khỏi danh sách ứng viên. Với `n` người, sau `n − 1` câu hỏi thích hợp chỉ còn **một** ứng viên.

In [5]:
def who_is_eliminated(a_knows_b, a, b):
    return a if a_knows_b else b


print("a biết b ->", who_is_eliminated(True, "A", "B"), "bị loại (A biết B nên A không phải người nổi tiếng)")
print("a không biết b ->", who_is_eliminated(False, "A", "B"), "bị loại (A không biết B nên B không phải người nổi tiếng)")

a biết b -> A bị loại (A biết B nên A không phải người nổi tiếng)
a không biết b -> B bị loại (A không biết B nên B không phải người nổi tiếng)


### 3.5. Kiểm chứng ứng viên: hai điều kiện, hai chiều

Một ứng viên `c` còn lại sau khi loại chưa chắc là người nổi tiếng (có thể **không ai** là người nổi tiếng). Phải **kiểm chứng** với mọi `i ≠ c`: (1) `c` **không** biết `i` (`not knows(c, i)`) và (2) `i` **biết** `c` (`knows(i, c)`). Chỉ cần một điều kiện vi phạm thì trả `-1`.

In [6]:
def verify(candidate, n, knows):
    for i in range(n):
        if i != candidate:
            if knows(candidate, i) or not knows(i, candidate):
                return False
    return True


for graph, expected in examples:
    p = Party(graph)
    print(expected, [c for c in range(len(graph)) if verify(c, len(graph), p.knows)])

1 [1]
-1 []


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Ví dụ 1: `graph = [[1,1,0],[0,1,0],[1,1,1]]`: `0 biết 1`, `1 không biết ai`, `2 biết 0 và 1`.

**Loại dần:** bắt đầu ứng viên là `0`.

- Hỏi `knows(0, 1)`: **có** → `0` bị loại; ứng viên mới là `1`.
- Hỏi `knows(1, 2)`: **không** → `2` bị loại; ứng viên vẫn là `1`.

Chỉ còn `1`. **Kiểm chứng** `1`: `knows(1, 0)` không (đúng, `1` không biết `0`), `knows(0, 1)` có (đúng, `0` biết `1`); `knows(1, 2)` không (đúng), `knows(2, 1)` có (đúng). Vậy người nổi tiếng là `1`.

Ví dụ 2: `graph = [[1,0,1],[1,1,0],[0,1,1]]`: ứng viên `0`; `knows(0,1) = 0` → `1` bị loại, giữ `0`; `knows(0,2) = 1` → `0` bị loại, ứng viên `2`. Kiểm chứng `2`: `knows(2, 0) = 0` — đúng; `knows(0, 2) = 1` — đúng; `knows(2, 1) = 1` — **sai** (người nổi tiếng không được biết ai). Trả `-1`.

In [7]:
for graph, expected in examples:
    p = Party(graph)
    n = len(graph)
    candidate = 0
    for i in range(1, n):
        a = p.knows(candidate, i)
        print(f"  knows({candidate}, {i}) = {a} -> {'ứng viên đổi thành ' + str(i) if a else 'giữ ứng viên ' + str(candidate)}")
        if a:
            candidate = i
    print(f"ứng viên cuối = {candidate}; kiểm chứng:", verify(candidate, n, p.knows), "| kỳ vọng:", expected)
    print()

  knows(0, 1) = True -> ứng viên đổi thành 1
  knows(1, 2) = False -> giữ ứng viên 1
ứng viên cuối = 1; kiểm chứng: True | kỳ vọng: 1

  knows(0, 1) = False -> giữ ứng viên 0
  knows(0, 2) = True -> ứng viên đổi thành 2
ứng viên cuối = 2; kiểm chứng: False | kỳ vọng: -1



## 5. Cách 1 — Brute Force / baseline: kiểm tra từng người làm ứng viên

**Ý tưởng.** Với mỗi `c` từ `0` đến `n − 1`, kiểm tra trực tiếp định nghĩa: `c` không biết ai khác và ai khác cũng biết `c`. Trả `c` đầu tiên thỏa (có nhiều nhất một). Dừng sớm khi một điều kiện vi phạm.

In [8]:
def celebrity_bruteforce(n: int, knows) -> int:
    for c in range(n):
        ok = True
        for i in range(n):
            if i == c:
                continue
            if knows(c, i) or not knows(i, c):     # c biết i, hoặc i không biết c -> c không phải người nổi tiếng
                ok = False
                break
        if ok:
            return c
    return -1


for graph, expected in examples:
    p = Party(graph)
    print(f"-> {celebrity_bruteforce(len(graph), p.knows)} (kỳ vọng {expected}), số lần gọi = {p.calls}")

-> 1 (kỳ vọng 1), số lần gọi = 5
-> -1 (kỳ vọng -1), số lần gọi = 7


**Độ phức tạp (Cách 1).** Mỗi ứng viên tốn tối đa `2(n − 1)` lần gọi `knows`, có `n` ứng viên: **`O(n²)` lần gọi**, tệ nhất `2n(n − 1)`. Với `n = 100` là khoảng `20000` lần: **vượt xa** giới hạn `3n = 300`. Thời gian cũng `O(n²)`, Space `O(1)`.

**Nút thắt.** Mỗi lần ta chỉ dùng câu trả lời để loại **một** ứng viên (nếu vi phạm) nhưng không khai thác thông tin mà câu trả lời còn cho biết về người kia. Mỗi câu hỏi có thể loại một người khỏi **hai** người được hỏi.

## 6. Ý tưởng thuật toán / bước đột phá

**Loại dần bằng một lượt.** Giữ một ứng viên `candidate`, ban đầu `0`. Với `i = 1, 2, ..., n − 1`: hỏi `knows(candidate, i)`.

- **Có:** `candidate` biết `i` nên `candidate` không phải người nổi tiếng; ứng viên mới là `i`.
- **Không:** `i` không được `candidate` biết nên `i` không phải người nổi tiếng; giữ `candidate`.

Sau `n − 1` câu hỏi chỉ còn **một** ứng viên: **mọi người khác đã bị loại**. Quan trọng: nếu **có** người nổi tiếng `c` thì `c` **không bao giờ bị loại** (không có lần nào `c` biết người khác; và nếu `c` được hỏi như `i` thì `knows(candidate, c)` luôn **có**, nên `c` thành ứng viên) nên ứng viên cuối phải là `c`. Nếu **không có** người nổi tiếng thì ứng viên cuối chưa chắc đúng.

Do đó cần **lượt kiểm chứng** (mục 3.5): với mọi `i ≠ candidate`, kiểm tra `not knows(candidate, i)` và `knows(i, candidate)`.

**Số lần gọi:** lượt loại `n − 1`, lượt kiểm chứng tối đa `2(n − 1)`: tổng tối đa `3(n − 1) < 3n`. Đạt follow up.

In [9]:
def elimination_demo(graph):
    p = Party(graph)
    n = len(graph)
    candidate = 0
    for i in range(1, n):
        if p.knows(candidate, i):
            candidate = i
    ok = verify(candidate, n, p.knows)
    return (candidate if ok else -1), p.calls


for graph, expected in examples:
    result, calls = elimination_demo(graph)
    print(f"kết quả {result} (kỳ vọng {expected}), số lần gọi {calls} <= 3n = {3 * len(graph)}")

kết quả 1 (kỳ vọng 1), số lần gọi 6 <= 3n = 9
kết quả -1 (kỳ vọng -1), số lần gọi 5 <= 3n = 9


## 7. Các cách tiếp theo

### Cách 2 — Đếm bậc: hỏi mọi cặp

**Giải thích.** Hỏi `knows(i, j)` cho mọi cặp `i ≠ j` (`n(n − 1)` lần); ghi `out_deg[i]` (số người `i` biết) và `in_deg[j]` (số người biết `j`). Người nổi tiếng là người có `out_deg = 0` và `in_deg = n − 1`. Dễ nghĩ nhưng luôn tốn `n(n − 1)` lần gọi.

In [10]:
def celebrity_degrees(n: int, knows) -> int:
    out_deg = [0] * n
    in_deg = [0] * n
    for i in range(n):
        for j in range(n):
            if i != j and knows(i, j):
                out_deg[i] += 1
                in_deg[j] += 1
    for c in range(n):
        if out_deg[c] == 0 and in_deg[c] == n - 1:
            return c
    return -1


for graph, expected in examples:
    p = Party(graph)
    print(f"-> {celebrity_degrees(len(graph), p.knows)} (kỳ vọng {expected}), số lần gọi = {p.calls} = n(n-1) = {len(graph) * (len(graph) - 1)}")

-> 1 (kỳ vọng 1), số lần gọi = 6 = n(n-1) = 6
-> -1 (kỳ vọng -1), số lần gọi = 6 = n(n-1) = 6


**Độ phức tạp Cách 2:** **`n(n − 1)` lần gọi** `knows`, Time `O(n²)`, Space `O(n)`. Vượt giới hạn `3n`.

### Cách 3 — Loại dần ứng viên rồi kiểm chứng (cách của lời giải tham khảo)

**Giải thích.** Hai lượt như mục 6: lượt 1 tìm ứng viên `ans` (nếu `knows(ans, i)` thì `ans = i`); lượt 2 kiểm chứng với mọi `i ≠ ans`: nếu `knows(ans, i)` hoặc `not knows(i, ans)` thì trả `-1`. Trả `ans`.

In [11]:
def celebrity_elimination(n: int, knows) -> int:
    ans = 0
    for i in range(1, n):                              # lượt 1: loại dần, còn một ứng viên
        if knows(ans, i):
            ans = i
    for i in range(n):                                 # lượt 2: kiểm chứng cả hai chiều
        if i != ans:
            if knows(ans, i) or not knows(i, ans):
                return -1
    return ans


for graph, expected in examples:
    p = Party(graph)
    print(f"-> {celebrity_elimination(len(graph), p.knows)} (kỳ vọng {expected}), số lần gọi = {p.calls} <= 3n = {3 * len(graph)}")

-> 1 (kỳ vọng 1), số lần gọi = 6 <= 3n = 9
-> -1 (kỳ vọng -1), số lần gọi = 5 <= 3n = 9


In [12]:
# Code nộp LeetCode cho Cách 3 (knows là hàm toàn cục do hệ thống chấm cung cấp)
class Solution:
    def findCelebrity(self, n: int) -> int:
        ans = 0
        for i in range(1, n):
            if knows(ans, i):
                ans = i
        for i in range(n):
            if ans != i:
                if knows(ans, i) or not knows(i, ans):
                    return -1
        return ans

**Độ phức tạp Cách 3:** tối đa `(n − 1) + 2(n − 1) = 3(n − 1)` lần gọi `knows`: **`O(n)` lần gọi**, Time `O(n)`, Space `O(1)`.

### Cách 4 — Ngăn xếp (stack) loại ứng viên

**Giải thích.** Đẩy mọi người `0..n−1` vào một ngăn xếp. Lặp: lấy ra hai người `a`, `b`; nếu `knows(a, b)` thì `a` bị loại, đẩy lại `b`; ngược lại `b` bị loại, đẩy lại `a`. Còn lại một người thì kiểm chứng như trước. Số câu hỏi để loại: `n − 1`.

In [13]:
def celebrity_stack(n: int, knows) -> int:
    stack = list(range(n))
    while len(stack) > 1:
        a = stack.pop()
        b = stack.pop()
        if knows(a, b):
            stack.append(b)                            # a biết b: a bị loại
        else:
            stack.append(a)                            # a không biết b: b bị loại
    candidate = stack[0]
    if verify(candidate, n, knows):
        return candidate
    return -1


for graph, expected in examples:
    p = Party(graph)
    print(f"-> {celebrity_stack(len(graph), p.knows)} (kỳ vọng {expected}), số lần gọi = {p.calls} <= 3n = {3 * len(graph)}")

-> 1 (kỳ vọng 1), số lần gọi = 6 <= 3n = 9
-> -1 (kỳ vọng -1), số lần gọi = 5 <= 3n = 9


**Độ phức tạp Cách 4:** **`≤ 3(n − 1)` lần gọi**, Time `O(n)`, Space `O(n)` cho ngăn xếp.

### Cách 5 — Hai con trỏ từ hai đầu

**Giải thích.** Đặt `l = 0`, `r = n − 1` là hai ứng viên. Trong khi `l < r`: nếu `knows(l, r)` thì `l` bị loại (`l += 1`); ngược lại `r` bị loại (`r -= 1`). Khi `l == r` đó là ứng viên cuối; kiểm chứng. Mỗi câu hỏi loại một người nên đúng `n − 1` câu cho lượt loại.

In [14]:
def celebrity_two_pointers(n: int, knows) -> int:
    l, r = 0, n - 1
    while l < r:
        if knows(l, r):
            l += 1                                     # l biết r: l bị loại
        else:
            r -= 1                                     # l không biết r: r bị loại
    if verify(l, n, knows):
        return l
    return -1


for graph, expected in examples:
    p = Party(graph)
    print(f"-> {celebrity_two_pointers(len(graph), p.knows)} (kỳ vọng {expected}), số lần gọi = {p.calls} <= 3n = {3 * len(graph)}")

-> 1 (kỳ vọng 1), số lần gọi = 6 <= 3n = 9
-> -1 (kỳ vọng -1), số lần gọi = 3 <= 3n = 9


**Độ phức tạp Cách 5:** **`≤ 3(n − 1)` lần gọi**, Time `O(n)`, Space `O(1)`.

> Cách 3 là cách nộp bài. Cách 4 và 5 là hai cách loại ứng viên khác. Cách 1 và 2 cho thấy số lần gọi `O(n²)` và là chuẩn đối chiếu.

## 8. TRACE từng bước

Ta theo dõi Cách 3 (loại dần + kiểm chứng) trên cả hai ví dụ, in từng lần gọi `knows` và ứng viên hiện tại.

In [15]:
def trace_elimination(graph):
    p = Party(graph)
    n = len(graph)
    print(f"graph={graph}")
    ans = 0
    print("  lượt 1 (loại dần):")
    for i in range(1, n):
        got = p.knows(ans, i)
        print(f"    knows({ans}, {i}) = {got} -> {'ứng viên = ' + str(i) if got else 'giữ ứng viên ' + str(ans)}")
        if got:
            ans = i
    print(f"  ứng viên sau lượt 1: {ans}")
    print("  lượt 2 (kiểm chứng):")
    for i in range(n):
        if i != ans:
            a = p.knows(ans, i)
            print(f"    knows({ans}, {i}) = {a} (cần False)")
            if a:
                print("    => ứng viên biết người khác: -1")
                return -1, p.calls
            b = p.knows(i, ans)
            print(f"    knows({i}, {ans}) = {b} (cần True)")
            if not b:
                print("    => có người không biết ứng viên: -1")
                return -1, p.calls
    print(f"  => người nổi tiếng: {ans}")
    return ans, p.calls


r1 = trace_elimination(examples[0][0])
assert r1 == (1, 6)
print()
r2 = trace_elimination(examples[1][0])
assert r2[0] == -1

graph=[[1, 1, 0], [0, 1, 0], [1, 1, 1]]
  lượt 1 (loại dần):
    knows(0, 1) = True -> ứng viên = 1
    knows(1, 2) = False -> giữ ứng viên 1
  ứng viên sau lượt 1: 1
  lượt 2 (kiểm chứng):
    knows(1, 0) = False (cần False)
    knows(0, 1) = True (cần True)
    knows(1, 2) = False (cần False)
    knows(2, 1) = True (cần True)
  => người nổi tiếng: 1

graph=[[1, 0, 1], [1, 1, 0], [0, 1, 1]]
  lượt 1 (loại dần):
    knows(0, 1) = False -> giữ ứng viên 0
    knows(0, 2) = True -> ứng viên = 2
  ứng viên sau lượt 1: 2
  lượt 2 (kiểm chứng):
    knows(2, 0) = False (cần False)
    knows(0, 2) = True (cần True)
    knows(2, 1) = True (cần False)
    => ứng viên biết người khác: -1


Bảng tóm tắt (ví dụ 1, `graph = [[1,1,0],[0,1,0],[1,1,1]]`, tổng `6` lần gọi `≤ 3n = 9`):

| lượt | câu hỏi | trả lời | kết luận |
|---|---|---|---|
| 1 | `knows(0, 1)` | có | `0` bị loại, ứng viên `1` |
| 1 | `knows(1, 2)` | không | `2` bị loại, giữ `1` |
| 2 | `knows(1, 0)` | không (đúng) | |
| 2 | `knows(0, 1)` | có (đúng) | `0` biết `1` |
| 2 | `knows(1, 2)` | không (đúng) | |
| 2 | `knows(2, 1)` | có (đúng) | `2` biết `1` → trả `1` |

## 9. Code nộp LeetCode

Bản nộp dùng Cách 3. Đúng class `Solution`, đúng method `findCelebrity(self, n: int) -> int`; `knows` là hàm toàn cục do hệ thống cung cấp (ở đây ta gán `knows` toàn cục bằng `party.knows` rồi gọi).

In [16]:
class Solution:
    def findCelebrity(self, n: int) -> int:
        ans = 0                                         # ứng viên hiện tại
        for i in range(1, n):
            if knows(ans, i):                           # ans biết i: ans không phải người nổi tiếng
                ans = i
        for i in range(n):                              # kiểm chứng ứng viên cuối
            if ans != i:
                if knows(ans, i) or not knows(i, ans):  # ans biết ai đó, hoặc có người không biết ans
                    return -1
        return ans


def run_solution(graph):
    """Gán knows toàn cục (như hệ thống chấm) rồi gọi Solution; trả (kết quả, số lần gọi)."""
    global knows
    party = Party(graph)
    knows = party.knows
    return Solution().findCelebrity(len(graph)), party.calls


for graph, expected in examples:
    result, calls = run_solution(graph)
    assert result == expected and calls <= 3 * len(graph)
print("PASS ví dụ đề cho class Solution (và số lần gọi <= 3n)")

PASS ví dụ đề cho class Solution (và số lần gọi <= 3n)


## 10. Vì sao thuật toán đúng

- **Người nổi tiếng nếu có thì không bao giờ bị loại ở lượt 1:** gọi `c` là người nổi tiếng. `knows(c, i)` luôn **không** (c không biết ai), nên `c` không bị loại khi là `ans`. Khi `c` được xét như `i`, `knows(ans, c)` luôn **có** (mọi người biết `c`), nên `ans` được đặt thành `c`. Vậy sau lượt 1, nếu có người nổi tiếng thì `ans = c`.
- **Ứng viên cuối là duy nhất có thể:** mỗi `knows(ans, i)` loại đúng một trong `ans`, `i` (mục 3.4); sau `n − 1` câu, chỉ còn một ứng viên. Mọi người khác đã bị loại khỏi khả năng làm người nổi tiếng.
- **Kiểm chứng đầy đủ:** `ans` là người nổi tiếng ⇔ mọi `i ≠ ans` thỏa `not knows(ans, i)` và `knows(i, ans)`. Lượt 2 kiểm tra đúng điều này; nếu có vi phạm thì **không ai** là người nổi tiếng (vì chỉ `ans` còn khả năng), trả `-1`.
- **Số lần gọi:** lượt 1 đúng `n − 1`; lượt 2 tối đa `2(n − 1)`; tổng `≤ 3(n − 1) < 3n`.
- **Cách 4 và 5** dùng cùng nguyên lý loại dần (mỗi câu loại một người) rồi kiểm chứng; chỉ khác thứ tự chọn cặp để hỏi.

## 11. Phân tích Time / Space Complexity

Gọi `n` là số người. Ta đo cả **số lần gọi `knows`** (đại lượng chính của bài) lẫn thời gian/bộ nhớ.

- **Cách 1 (kiểm tra từng ứng viên):** số lần gọi `O(n²)` (tối đa `2n(n − 1)`), Time `O(n²)`, Space `O(1)`.
- **Cách 2 (đếm bậc mọi cặp):** đúng `n(n − 1)` lần gọi, Time `O(n²)`, Space `O(n)`.
- **Cách 3 (loại dần + kiểm chứng):** `≤ 3(n − 1)` lần gọi, Time `O(n)`, Space `O(1)`.
- **Cách 4 (ngăn xếp):** `≤ 3(n − 1)` lần gọi, Time `O(n)`, Space `O(n)`.
- **Cách 5 (hai con trỏ):** `≤ 3(n − 1)` lần gọi, Time `O(n)`, Space `O(1)`.
- Giới hạn dưới: cần ít nhất `Ω(n)` câu hỏi (để thấy từng người ít nhất một lần), nên `O(n)` là tối ưu về tiệm cận.

## 12. Bộ test

`oracle(graph)` đọc **trực tiếp ma trận**: người `c` là người nổi tiếng nếu `graph[c][j] == 0` với mọi `j ≠ c` và `graph[i][c] == 1` với mọi `i ≠ c` (không dùng `knows`). Đồ thị ngẫu nhiên gồm hai loại: **có người nổi tiếng** (xây rồi trộn nhãn) và **ngẫu nhiên hoàn toàn** (thường không có). Với các cách `O(n)`, ta kiểm tra thêm `số lần gọi ≤ 3n`.

In [17]:
sol = Solution()
impls = {
    "bruteforce": celebrity_bruteforce,
    "degrees": celebrity_degrees,
    "elimination": celebrity_elimination,
    "stack": celebrity_stack,
    "two_pointers": celebrity_two_pointers,
}
LINEAR = ("elimination", "stack", "two_pointers")


def oracle(graph):
    n = len(graph)
    found = [c for c in range(n)
             if all(graph[c][j] == 0 for j in range(n) if j != c) and all(graph[i][c] == 1 for i in range(n) if i != c)]
    assert len(found) <= 1
    return found[0] if found else -1


def make_graph(n, celebrity=None, density=0.5):
    """Đồ thị ngẫu nhiên; nếu celebrity được cho thì đặt người đó làm người nổi tiếng."""
    g = [[1 if i == j else int(random.random() < density) for j in range(n)] for i in range(n)]
    if celebrity is not None:
        for j in range(n):
            g[celebrity][j] = 1 if j == celebrity else 0
        for i in range(n):
            g[i][celebrity] = 1
    return g


def run_all(graph):
    results = {}
    n = len(graph)
    for name, fn in impls.items():
        p = Party(graph)
        results[name] = (fn(n, p.knows), p.calls)
    global knows
    p = Party(graph)
    knows = p.knows
    results["submit"] = (sol.findCelebrity(n), p.calls)
    return results


for graph, expected in examples:
    assert oracle(graph) == expected
    for name, (got, calls) in run_all(graph).items():
        assert got == expected, (name, graph, got, expected)
print("PASS ví dụ của đề (mọi cách và oracle đọc thẳng ma trận)")

PASS ví dụ của đề (mọi cách và oracle đọc thẳng ma trận)


### 12.1. Các trường hợp biên

In [18]:
def check_all(graph, expected, label):
    assert oracle(graph) == expected, (label, "oracle", oracle(graph), expected)
    n = len(graph)
    for name, (got, calls) in run_all(graph).items():
        assert got == expected, (label, name, got, expected)
        if name in LINEAR or name == "submit":
            assert calls <= 3 * n, (label, name, calls, 3 * n)
    print("PASS", label, "->", expected)


check_all([[1, 1], [0, 1]], 1, "n = 2: người 1 là người nổi tiếng")
check_all([[1, 0], [1, 1]], 0, "n = 2: người 0 là người nổi tiếng")
check_all([[1, 1], [1, 1]], -1, "n = 2: biết lẫn nhau, không ai")
check_all([[1, 0], [0, 1]], -1, "n = 2: không ai biết ai")
check_all([[1, 0, 0], [1, 1, 0], [1, 0, 1]], 0, "người nổi tiếng ở đầu (0)")
check_all([[1, 0, 1], [0, 1, 1], [0, 0, 1]], 2, "người nổi tiếng ở cuối (2)")
check_all([[1, 1, 0], [0, 1, 0], [1, 1, 1]], 1, "ví dụ 1 của đề")
check_all([[1, 0, 1], [1, 1, 0], [0, 1, 1]], -1, "ví dụ 2 của đề")
check_all([[1 if i == j else 1 for j in range(5)] for i in range(5)], -1, "mọi người biết mọi người")
check_all([[1 if i == j else 0 for j in range(5)] for i in range(5)], -1, "không ai biết ai (n = 5)")
n100 = 100
g = make_graph(n100, celebrity=57)
check_all(g, 57, "n = 100, người nổi tiếng ở giữa")
g = make_graph(n100, celebrity=0)
check_all(g, 0, "n = 100, người nổi tiếng là 0")
g = make_graph(n100, celebrity=99)
check_all(g, 99, "n = 100, người nổi tiếng là 99")

PASS n = 2: người 1 là người nổi tiếng -> 1
PASS n = 2: người 0 là người nổi tiếng -> 0
PASS n = 2: biết lẫn nhau, không ai -> -1
PASS n = 2: không ai biết ai -> -1
PASS người nổi tiếng ở đầu (0) -> 0
PASS người nổi tiếng ở cuối (2) -> 2
PASS ví dụ 1 của đề -> 1
PASS ví dụ 2 của đề -> -1
PASS mọi người biết mọi người -> -1
PASS không ai biết ai (n = 5) -> -1
PASS n = 100, người nổi tiếng ở giữa -> 57
PASS n = 100, người nổi tiếng là 0 -> 0
PASS n = 100, người nổi tiếng là 99 -> 99


### 12.2. Tính chất và phản ví dụ

In [19]:
random.seed(277)

# Tính chất 1: đồ thị có người nổi tiếng -> mọi cách tìm đúng, và các cách tuyến tính dùng <= 3(n - 1) lần gọi
for _ in range(300):
    n = random.randint(2, 30)
    c = random.randrange(n)
    graph = make_graph(n, celebrity=c)
    for name, (got, calls) in run_all(graph).items():
        assert got == c, (name, graph, got, c)
        if name in LINEAR or name == "submit":
            assert calls <= 3 * (n - 1), (name, calls, n)
print("PASS tính chất: có người nổi tiếng thì tìm đúng và số lần gọi <= 3(n - 1)")

# Tính chất 2: các cách O(n) gọi không quá 3(n - 1) lần kể cả khi không có người nổi tiếng
for _ in range(300):
    n = random.randint(2, 40)
    graph = make_graph(n, density=random.choice([0.2, 0.5, 0.8]))
    for name, (got, calls) in run_all(graph).items():
        if name in LINEAR or name == "submit":
            assert calls <= 3 * (n - 1)
print("PASS tính chất: số lần gọi <= 3(n - 1) với đồ thị bất kỳ")

# Tính chất 3: cách brute force/đếm bậc dùng nhiều gọi hơn rõ rệt khi n lớn
graph = make_graph(100, celebrity=99)
calls_brute = run_all(graph)["bruteforce"][1]
calls_degrees = run_all(graph)["degrees"][1]
calls_linear = run_all(graph)["elimination"][1]
print(f"n = 100: brute force {calls_brute} lần, đếm bậc {calls_degrees} lần, loại dần {calls_linear} lần")
assert calls_degrees == 100 * 99 and calls_linear <= 297 and calls_brute > calls_linear

# Phản ví dụ: không kiểm chứng sau khi chọn ứng viên
def sai_khong_kiem_chung(n, knows):
    ans = 0
    for i in range(1, n):
        if knows(ans, i):
            ans = i
    return ans

for graph, expected in examples:
    p = Party(graph)
    print(f"{graph} -> sai: {sai_khong_kiem_chung(len(graph), p.knows)} | đúng: {expected}")
p = Party(examples[1][0])
assert sai_khong_kiem_chung(3, p.knows) != -1

PASS tính chất: có người nổi tiếng thì tìm đúng và số lần gọi <= 3(n - 1)


PASS tính chất: số lần gọi <= 3(n - 1) với đồ thị bất kỳ


n = 100: brute force 394 lần, đếm bậc 9900 lần, loại dần 297 lần
[[1, 1, 0], [0, 1, 0], [1, 1, 1]] -> sai: 1 | đúng: 1
[[1, 0, 1], [1, 1, 0], [0, 1, 1]] -> sai: 2 | đúng: -1


### 12.3. Đối chiếu ngẫu nhiên (có seed) với oracle độc lập

In [20]:
random.seed(2026)
cases = 0
for trial in range(2500):
    n = random.randint(2, 12)
    kind = random.random()
    if kind < 0.4:
        graph = make_graph(n, celebrity=random.randrange(n))
    elif kind < 0.7:
        graph = make_graph(n, celebrity=random.randrange(n))
        # phá hỏng: thêm một cạnh từ người nổi tiếng hoặc bỏ một cạnh tới người đó -> thường không còn người nổi tiếng
        c = oracle(graph)
        if random.random() < 0.5:
            j = random.choice([x for x in range(n) if x != c])
            graph[c][j] = 1
        else:
            i = random.choice([x for x in range(n) if x != c])
            graph[i][c] = 0
    else:
        graph = make_graph(n, density=random.choice([0.2, 0.5, 0.9]))
    expected = oracle(graph)
    for name, (got, calls) in run_all(graph).items():
        assert got == expected, (name, graph, got, expected)
        if name in LINEAR or name == "submit":
            assert calls <= 3 * (n - 1)
    cases += 1
print("PASS", cases, "đồ thị ngẫu nhiên (có, không có, bị phá hỏng người nổi tiếng): mọi cách khớp oracle; cách O(n) <= 3(n - 1) lần gọi")

PASS 2500 đồ thị ngẫu nhiên (có, không có, bị phá hỏng người nổi tiếng): mọi cách khớp oracle; cách O(n) <= 3(n - 1) lần gọi


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Chọn ứng viên rồi trả luôn, không kiểm chứng

In [21]:
for graph, expected in examples:
    p = Party(graph)
    print(f"graph có đáp án {expected} -> sai (không kiểm chứng): {sai_khong_kiem_chung(len(graph), p.knows)}")

graph có đáp án 1 -> sai (không kiểm chứng): 1
graph có đáp án -1 -> sai (không kiểm chứng): 2


**Vì sao sai:** lượt loại dần luôn trả về **một ứng viên**, kể cả khi **không có** người nổi tiếng nào. Ví dụ 2 cho ứng viên `2` nhưng đáp án là `-1`. Cần lượt kiểm chứng.

### Lỗi 2: Chỉ kiểm chứng một chiều

In [22]:
def sai_mot_chieu(n, knows):
    ans = 0
    for i in range(1, n):
        if knows(ans, i):
            ans = i
    for i in range(n):
        if i != ans and knows(ans, i):                # chỉ kiểm tra "ans không biết ai"
            return -1
    return ans


graph = [[1, 0, 0], [0, 1, 0], [0, 0, 1]]             # không ai biết ai: không có người nổi tiếng
p = Party(graph)
print("sai:", sai_mot_chieu(3, p.knows), "| đúng:", oracle(graph))

sai: 0 | đúng: -1


**Vì sao sai:** người nổi tiếng cần **cả hai** điều kiện: không biết ai **và** ai cũng biết. Nếu chỉ kiểm tra "không biết ai" thì ứng viên cuối của đồ thị không ai biết ai vẫn được nhận. Phải thêm `knows(i, ans)`.

### Lỗi 3: Cập nhật ứng viên sai hướng

In [23]:
def sai_huong(n, knows):
    ans = 0
    for i in range(1, n):
        if not knows(ans, i):                          # sai: nếu ans không biết i thì i bị loại, không phải ans
            ans = i
    for i in range(n):
        if ans != i and (knows(ans, i) or not knows(i, ans)):
            return -1
    return ans


p = Party(examples[0][0])
print("ví dụ 1 -> sai:", sai_huong(3, p.knows), "| đúng:", examples[0][1])

ví dụ 1 -> sai: -1 | đúng: 1


**Vì sao sai:** nếu `ans` **biết** `i` thì `ans` bị loại (người nổi tiếng không biết ai) nên ứng viên mới là `i`. Nếu `ans` **không biết** `i` thì `i` bị loại. Đảo hướng loại nhầm người nổi tiếng thật.

### Lỗi 4: Hỏi `knows(i, i)` (một người có biết chính mình)

In [24]:
def sai_hoi_chinh_minh(n, knows_raw):
    ans = 0
    for i in range(1, n):
        if knows_raw(ans, i):
            ans = i
    for i in range(n):                                 # sai: không bỏ qua i == ans
        if knows_raw(ans, i) or (i != ans and not knows_raw(i, ans)):
            return -1
    return ans


graph = examples[0][0]
raw = lambda a, b: bool(graph[a][b])                   # hàm thô không cấm hỏi (a, a)
print("ví dụ 1 -> sai:", sai_hoi_chinh_minh(3, raw), "| đúng:", examples[0][1])

ví dụ 1 -> sai: -1 | đúng: 1


**Vì sao sai:** theo đề `graph[i][i] = 1` (ai cũng "biết" chính mình), nên `knows(ans, ans)` luôn `True` và loại cả người nổi tiếng thật. Phải bỏ qua `i == ans` ở lượt kiểm chứng (và không bao giờ hỏi một người về chính họ).

## 14. Mẹo làm nhanh & nhận dạng pattern

- Thấy "tìm một phần tử đặc biệt bằng số truy vấn ít nhất": tìm **phép loại** mà **mỗi truy vấn loại một ứng viên** → lượt loại `O(n)` + lượt kiểm chứng `O(n)`.
- Quy tắc loại: nếu `knows(a, b)` thì loại `a`, ngược lại loại `b`.
- **Luôn kiểm chứng** ứng viên cuối nếu đề không đảm bảo sự tồn tại.
- Ghi nhớ hai điều kiện của người nổi tiếng: **không biết ai** *và* **ai cũng biết**.
- Cùng ý tưởng với "tìm phần tử chiếm đa số" (Boyer–Moore): triệt tiêu cặp, kiểm chứng cuối.
- Test bài tương tác: dựng `Party` đếm số lần gọi; kiểm tra giới hạn `3n` bằng `assert`.

## 15. Biến thể của bài

1. **Có thể không có người nổi tiếng nhưng giới hạn gọi `2n`** (cần cách loại hiệu quả hơn?).
2. **Nhiều người nổi tiếng** (tập con nổi tiếng): định nghĩa khác.
3. **Tìm người "bị mọi người biết"** (bậc vào `n − 1`, không yêu cầu không biết ai).
4. **Đồ thị trọng số / quan hệ không đối xứng khác** (tìm "sink" trong đồ thị có hướng — **universal sink**).
5. **Tìm celebrity trực tiếp từ ma trận** `O(n)` thao tác (tìm "sink" trong ma trận kề).

In [25]:
def universal_sink_from_matrix(graph):
    """Biến thể 4/5: tìm 'sink' toàn năng trong ma trận kề (bỏ qua đường chéo) với O(n) thao tác đọc ma trận."""
    n = len(graph)
    i = j = 0
    # tìm ứng viên: nếu graph[i][j] = 1 (i biết j) thì i bị loại -> i += 1; ngược lại j bị loại -> j += 1
    candidate = 0
    for k in range(1, n):
        if graph[candidate][k]:
            candidate = k
    if all(graph[candidate][k] == 0 for k in range(n) if k != candidate) and all(graph[k][candidate] == 1 for k in range(n) if k != candidate):
        return candidate
    return -1


def count_known_by_all(n, knows):
    """Biến thể 3 (đơn giản, O(n^2)): số người mà mọi người khác đều biết."""
    return sum(all(knows(i, c) for i in range(n) if i != c) for c in range(n))


def two_celebrities_possible_demo():
    """Biến thể 2: với định nghĩa 'người nổi tiếng' như đề, không thể có hai người (đã chứng minh)."""
    g = [[1, 1, 1], [1, 1, 1], [1, 1, 1]]
    return oracle(g)


random.seed(15)
for _ in range(300):
    n = random.randint(2, 10)
    g = make_graph(n, celebrity=random.choice([None, random.randrange(n)]))
    assert universal_sink_from_matrix(g) == oracle(g)
    p = Party(g)
    cnt = count_known_by_all(n, p.knows)
    assert cnt >= (1 if oracle(g) != -1 else 0)
assert two_celebrities_possible_demo() == -1
print("PASS biến thể: sink toàn năng từ ma trận, đếm người được mọi người biết, hai người nổi tiếng không thể xảy ra")

PASS biến thể: sink toàn năng từ ma trận, đếm người được mọi người biết, hai người nổi tiếng không thể xảy ra


## 16. So sánh các cách làm

| Cách | Ý tưởng | Số lần gọi `knows` | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Kiểm tra từng ứng viên | kiểm tra định nghĩa với mọi `c` | `O(n²)` | `O(1)` | hiểu đề; oracle |
| 2. Đếm bậc mọi cặp | `out_deg = 0`, `in_deg = n − 1` | đúng `n(n − 1)` | `O(n)` | dễ nghĩ, không tối ưu |
| 3. Loại dần + kiểm chứng | `knows(ans, i)` thì `ans = i`; rồi kiểm chứng | `≤ 3(n − 1)` | `O(1)` | **cách nộp bài** (follow up) |
| 4. Ngăn xếp | lấy hai người, loại một | `≤ 3(n − 1)` | `O(n)` | minh họa "mỗi câu loại một người" |
| 5. Hai con trỏ | `l`, `r` từ hai đầu | `≤ 3(n − 1)` | `O(1)` | gọn, cũng `O(1)` bộ nhớ |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán với `graph = [[1,0],[1,1]]`: người nổi tiếng là ai? và `Solution().findCelebrity` cho `[[1,1],[1,1]]` là gì? Số người nổi tiếng tối đa trong một bữa tiệc là bao nhiêu?

**Bài 2 (tính tay).** Với `graph = [[1,0,0,1],[1,1,0,1],[1,1,1,1],[0,0,0,1]]`: chạy lượt loại dần (ghi từng câu hỏi), rồi kiểm chứng; kết luận người nổi tiếng. Làm lại sau khi đổi `graph[3][0] = 1`.

**Bài 3 (điền TODO).** Hoàn thành `celebrity_todo(n, knows)` theo Cách 3.

In [26]:
def celebrity_todo(n, knows):
    ans = 0
    # TODO 1: for i in range(1, n): nếu knows(ans, i) thì ans = i
    # TODO 2: for i in range(n): bỏ qua i == ans; nếu knows(ans, i) hoặc not knows(i, ans) thì trả -1
    # TODO 3: trả ans
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `celebrity_stack2(n, knows)` theo Cách 4 (ngăn xếp). Kiểm tra bằng hai ví dụ của đề (qua `Party`) và đếm số lần gọi `≤ 3n`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `celebrity_with_call_limit(n, knows, limit)` trả về `(kết quả, số_lần_gọi)` và `raise` nếu vượt `limit`. (b) Viết `find_all_known_by_everyone(n, knows)` trả về danh sách **mọi** người mà tất cả người khác đều biết (bậc vào `n − 1`), không yêu cầu họ không biết ai.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: người nổi tiếng được mọi người biết và không biết ai; nếu hai người biết lẫn nhau thì không thể cùng là người nổi tiếng. Bài 2: nếu `a` biết `b` thì loại `a`, ngược lại loại `b`.
- **Mức 2 (vừa):** Bài 1: `graph = [[1,0],[1,1]]`: người `0` (người `1` biết `0`, `0` không biết `1`): người nổi tiếng là `0`; `[[1,1],[1,1]]`: `-1`; tối đa một người. Bài 2: ứng viên `0`: `knows(0,1)=0` giữ `0`; `knows(0,2)=0` giữ `0`; `knows(0,3)=1` → ứng viên `3`. Kiểm chứng `3`: `knows(3,0)=0`, `knows(0,3)=1`; `knows(3,1)=0`, `knows(1,3)=1`; `knows(3,2)=0`, `knows(2,3)=1`: người nổi tiếng `3`. Sau khi đặt `graph[3][0]=1`: `3` biết `0` nên `-1`. Bài 3: ba TODO trong comment.
- **Mức 3 (sâu):** Bài 4: đẩy `0..n−1`; `a = pop()`, `b = pop()`; `knows(a, b)` thì đẩy `b` ngược lại đẩy `a`; cuối cùng kiểm chứng. Bài 5(a): bọc `knows` bằng bộ đếm và `raise RuntimeError` khi vượt `limit`. Bài 5(b): hỏi mọi cặp `knows(i, c)`; hoặc dùng `Party` đọc cả ma trận (`O(n²)` gọi).

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [27]:
# Bài 1
assert oracle([[1, 0], [1, 1]]) == 0
assert oracle([[1, 1], [1, 1]]) == -1

# Bài 2
g2 = [[1, 0, 0, 1], [1, 1, 0, 1], [1, 1, 1, 1], [0, 0, 0, 1]]
assert oracle(g2) == 3
p = Party(g2)
assert celebrity_elimination(4, p.knows) == 3 and p.calls <= 12
g2b = [row[:] for row in g2]
g2b[3][0] = 1
assert oracle(g2b) == -1 and celebrity_elimination(4, Party(g2b).knows) == -1

# Bài 3
def celebrity_todo_ans(n, knows):
    ans = 0
    for i in range(1, n):
        if knows(ans, i):
            ans = i
    for i in range(n):
        if i == ans:
            continue
        if knows(ans, i) or not knows(i, ans):
            return -1
    return ans

for graph, expected in examples:
    assert celebrity_todo_ans(len(graph), Party(graph).knows) == expected

# Bài 4
def celebrity_stack2(n, knows):
    stack = list(range(n))
    while len(stack) > 1:
        a, b = stack.pop(), stack.pop()
        stack.append(b if knows(a, b) else a)
    c = stack[0]
    for i in range(n):
        if i != c and (knows(c, i) or not knows(i, c)):
            return -1
    return c

for graph, expected in examples:
    p = Party(graph)
    assert celebrity_stack2(len(graph), p.knows) == expected and p.calls <= 3 * len(graph)

# Bài 5(a), 5(b)
def celebrity_with_call_limit(n, knows, limit):
    calls = 0

    def counted(a, b):
        nonlocal calls
        calls += 1
        if calls > limit:
            raise RuntimeError("vượt quá giới hạn số lần gọi knows")
        return knows(a, b)

    return celebrity_elimination(n, counted), calls

def find_all_known_by_everyone(n, knows):
    return [c for c in range(n) if all(knows(i, c) for i in range(n) if i != c)]

r, calls = celebrity_with_call_limit(3, Party(examples[0][0]).knows, 9)
assert r == 1 and calls == 6
try:
    celebrity_with_call_limit(3, Party(examples[0][0]).knows, 3)
    raise AssertionError("đáng lẽ phải vượt giới hạn")
except RuntimeError:
    pass
random.seed(19)
for _ in range(100):
    n = random.randint(2, 8)
    g = make_graph(n, celebrity=random.choice([None, random.randrange(n)]))
    assert find_all_known_by_everyone(n, Party(g).knows) == [c for c in range(n) if all(g[i][c] for i in range(n) if i != c)]
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (loại dần ứng viên, rồi kiểm chứng cả hai chiều)

```python
class Solution:
    def findCelebrity(self, n: int) -> int:
        ans = 0
        for i in range(1, n):
            if knows(ans, i):
                ans = i
        for i in range(n):
            if ans != i:
                if knows(ans, i) or not knows(i, ans):
                    return -1
        return ans
```

**Checklist**

- [ ] Quy tắc loại đúng: `knows(ans, i)` → `ans = i` (loại `ans`); không biết → giữ `ans` (loại `i`)?
- [ ] Có lượt **kiểm chứng** sau khi chọn ứng viên (đề không đảm bảo có người nổi tiếng)?
- [ ] Kiểm chứng **cả hai chiều**: `not knows(ans, i)` **và** `knows(i, ans)`?
- [ ] Bỏ qua `i == ans`; không bao giờ gọi `knows(i, i)`?
- [ ] Số lần gọi `knows` `≤ 3(n − 1)` (đã đếm trong test)?
- [ ] Đã thử: `n = 2`, người nổi tiếng ở đầu/cuối, không ai biết ai, mọi người biết mọi người?
- [ ] Đã đối chiếu với oracle đọc thẳng ma trận trên đồ thị ngẫu nhiên (có/không có người nổi tiếng)?
- [ ] Skeleton đúng `class Solution` / `def findCelebrity(self, n: int) -> int` và dùng hàm `knows(a, b)` toàn cục?